In [1]:
import os
import re
import time
import paramiko

from tqdm import tqdm
from dotenv import load_dotenv

# load environment variables
load_dotenv("../server/.env")

print(os.getenv("SSH_HOST"))


sm-server


In [2]:
#! read pubspec.yaml
content = ""
with open("pubspec.yaml", "r", encoding="utf-8") as f:
    content = f.read()


#! find the current build number
build_match = re.search(r"version: (\d+).(\d+).(\d+)\+(\d+)", content)
if build_match:

    major = int(build_match.group(1))
    print(f"major : {major}")

    minor = int(build_match.group(2))
    print(f"minor : {minor}")

    patch = int(build_match.group(3))
    print(f"patch : {patch}")

    build_num = int(build_match.group(4))
    new_build_num = build_num + 1
    print(f"build_num : {build_num} -> new_build_num : {new_build_num}")

    # update the build number in pubspec.yaml content
    new_content = re.sub(
        r"version: (\d+)\.(\d+)\.(\d+)\+(\d+)",
        f"version: {build_match.group(1)}.{build_match.group(2)}.{build_match.group(3)}+{new_build_num}",
        content,
    )
    # print(new_content)

    # write back to env.dart
    with open("pubspec.yaml", "w", encoding="utf-8") as f:
        f.write(new_content)




major : 1
minor : 0
patch : 0
build_num : 54 -> new_build_num : 55


In [3]:

# read lib/Environment.dart
with open("lib/Environment.dart", "r", encoding="utf-8") as f:
    env_content = f.read()
# print(env_content)

# change bool is_local = false;  to bool is_local = true;
env_content = re.sub(r"bool is_local = false;", "bool is_local = true;", env_content)

# write back to env.dart
with open("lib/Environment.dart", "w", encoding="utf-8") as f:
    f.write(env_content)



In [4]:

#! clean
# os.system("flutter clean")

#! build
os.system(f"flutter build web --release --base-href / --output=build/local --no-wasm-dry-run")

#! copy
os.system("xcopy build\\local\\* ..\\server\\service\\admin\\local\\ /E /I /Y")


0

In [5]:

#! delay
for _ in tqdm(range(100)):
    time.sleep(0.1)



#! git commit and push
os.chdir("../server")
os.system("git add .")
os.system('git commit -m "update"')
os.system("git push")
os.chdir("../admin")


100%|██████████| 100/100 [00:10<00:00,  9.89it/s]


In [6]:
#! delay for 10 seconds
for _ in tqdm(range(100), desc="Waiting"):
    time.sleep(0.01)


#! create SSH client
client = paramiko.SSHClient()
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

#! connect to the server
client.connect(
    hostname=os.getenv("SSH_HOST"),
    port=22,
    username=os.getenv("SSH_USERNAME"),
    password=os.getenv("SSH_PASSWORD"),
)


#! single line commands
command = [
    "cd /root/server",
    "git pull",
    "docker compose -f docker-compose.yml up -d --build admin",  # update admin only
]

#! execute commands
stdin, stdout, stderr = client.exec_command(" && ".join(command))
print(stdout.read().decode())


#! print success message
print("Update successfully!")

#! close the connection
client.close()

Waiting: 100%|██████████| 100/100 [00:01<00:00, 96.11it/s]


Updating 4d962d8..1503d0f
Fast-forward
 service/admin/local/flutter_bootstrap.js | 2 +-
 service/admin/local/main.dart.js         | 2 +-
 service/admin/local/version.json         | 2 +-
 3 files changed, 3 insertions(+), 3 deletions(-)
#0 building with "default" instance using docker driver

#1 [admin internal] load build definition from Dockerfile
#1 transferring dockerfile: 361B done
#1 DONE 0.0s

#2 [admin internal] load metadata for docker.io/library/nginx:stable-alpine
#2 DONE 0.9s

#3 [admin internal] load .dockerignore
#3 transferring context: 98B done
#3 DONE 0.0s

#4 [admin internal] load build context
#4 transferring context: 2.90MB 0.1s done
#4 DONE 0.1s

#5 [admin 1/4] FROM docker.io/library/nginx:stable-alpine@sha256:0272e4604ed93c1792f03695a033a6e8546840f86e0de20a884bb17d2c924883
#5 resolve docker.io/library/nginx:stable-alpine@sha256:0272e4604ed93c1792f03695a033a6e8546840f86e0de20a884bb17d2c924883 0.1s done
#5 DONE 0.1s

#6 [admin 2/4] RUN rm -rf /usr/share/nginx/html/*


In [7]:
#! change bool is_local = false;  to bool is_local = true;
env_content = re.sub(r"bool is_local = true;", "bool is_local = false;", env_content)

#! write back to env.dart
with open("lib/Environment.dart", "w", encoding="utf-8") as f:
    f.write(env_content)